### 219. Contains Duplicate II
Given an integer array `nums` and an integer `k`, return `true` if there are two distinct indices `i` and `j` in the array such that `nums[i] == nums[j]` and `abs(i - j) <= k`.

**Example 1:**
```shell
Input: nums = [1,2,3,1], k = 3
Output: true
```
**Example 2:**
```shell
Input: nums = [1,0,1,1], k = 1
Output: true
```
**Example 3:**
```shell
Input: nums = [1,2,3,1,2,3], k = 2
Output: false
```
**Constraints:**

- `1 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`
- `0 <= k <= 10^5`

In [1]:
def func(nums: list[int], k: int) -> bool:
    # brute force
    # setup
    # k = 0 -> False: stop early
    # scan input - for loop - full pass - set i
    # scan input - for loop [i + 1, len(nums)] - set j

    # if i = j & abs(i - j) <= k -> True
    # return False

    if k == 0: return False

    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j] and abs(i - j) <= k:
                return True
    return False

In [5]:
# func([1,2,3,1], 3)  # True
# func([1,2,3,4,5], 10)  # False
# func([1,0,1,1], 1)  # True
func([1,2,3,1,2,3], 2)  # False

False

### Reasoning
**Step 1 — restate the problem in your own words**, and ask the clarifying questions an interviewer would expect (bounds, empty/None, duplicates, sign of k, what happens at the edges, etc.).
- I have an integer array, length 1 to 100000, with negatives, zeros, duplicates within a range -109 to 10^9. Also given `k` integer value from 0 to 100000. I need to test it if there are **two different elements** that are duplicates and the **aboslute value of between their indices** `<= k`. I need to return `True` if the index distance is within `k` or `False` otherwise, or if there are no duplicates.

**Basically, I'm testing the array for**:
- are there duplicates, at all; if no - return `False` no matter what
- `abs(i - j)` - duplicates index distance is **within** `k` - `True`, otherwise `False`

What is the length of input array and `k` range? Are there negatives, duplicates and zeros possible in the array? Can the array be empty or null?

Answering as the interviewer:

- **Array length**: up to ~10⁵ elements.
- **k range**: `0 <= k <= 10⁵` — non-negative, and **note `k` can be `0`**.
- **Values**: can be negative, positive, or zero; duplicates allowed.
- **Empty/null**: no, assume the array is valid with at least 1 element.

Given `k` can be `0`, that's worth pausing on: what would `abs(i - j) <= 0` mean for two _distinct_ indices `i` and `j`? Is that even possible?
- Two distinct indices will always have absolute value `> 0`, although you can use both normal and negative indices at the same time but it makes no sense. `abs(i - j) <= 0 `always `False`, it cannot be zero or negative since we use absolute value which has no sign.

Right, that sub-question is resolved:
> with `k=0`, no valid pair can ever exist, so the function should always return `False` regardless of array content — worth keeping as an **explicit test case**, not just a mental note.

**Step 2 — trace one clean example by hand** (pick your own `nums` and `k`), state the expected output, then layer edge cases on top: what about `k=0`, and what about `k` larger than the array length?
- `[1,2,3,1], k = 3` is given. Traverse the array for duplicate, so here `nums[0] == nums[3]` and `abs(0-3) <= 3` returns `True`. 
- For `k = 0`: `abs(0-3) <= 0` returns `False`.
- For `k = 5`: `abs(0-3) <= 5` return `True`.
- We can state: for `k=0` the output will always be `False`, no such indices exist to satisfy this condition; for `k >= len(nums)-1` it always be `True` if duplicates exist.

[Visual](../images/219contains_duplicatesII_cases.png)

Good, the base example trace and the k=0 rule are correct. But check this generalization — "for k > len(nums) it's always True" — against a case where the array has **no duplicates at all**: `nums = [1,2,3,4,5]`, `k = 10`. Does that return `True`?

Walk through why. What does a large `k` actually guarantee, and what does it _not_ guarantee, about the answer?
- If no duplicates at all it should always be `False`, according task `i == j`. So, large `k` **only** guarantees return `True` if there are duplicates in the array.

[Visual](../images/219contains_duplicatesII_cases2.png)

Correct correction. To sharpen it further:
> once `k` is large enough that `abs(i-j) <= k` holds for _every possible_ pair of indices in the array — which happens once `k >= len(nums) - 1`, the maximum possible index distance — the distance constraint stops mattering at all, and the problem collapses exactly into **Contains Duplicate I**: does any duplicate value exist anywhere in the array. Nice connection to draw for pattern recognition — Contains Duplicate II is really "Contains Duplicate I with a sliding window constraint on **how far apart the duplicate is allowed to be**."

[Visaul](../images/219contains_duplicatesII_pattern_recognition.png)

**Step 3 — brute force**.

What's the naive nested-loop approach here, and what's its time complexity?
- The naive nested-loop approach is check every possible pair for duplicates and if they found their indices difference is within given k - returns True, otherwise False. Could use if `k == 0`: return `False` filter at the beginning to stop early.
- If there are no duplicates it will search through every possible pair, returning False ultimately - worst case n ops for n items wich is $O(n^2)$ quadratic time. Possibly if duplicates are at the beginning of the data set and the indices within `k` it could be faster practically for small data sets, but I still dealling with 100000 items I tend to $O(n^2)$ any ways. Extra space is not used - space is $O(1)$.

[Visual](../images/219contains_duplicatesII_brute.png)
```py
def func(nums: list[int]) -> bool:
    if k == 0:
        return False

    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j] and abs(i - j) <= k:
                return True
    return False
```

**Step 4 — select algorithm + state complexity**

What's the redundant work in that $O(n²)$ approach**, what data structure collapses it, and what's the target time/space complexity plus the one-sentence key insight?
- The redundant work in the brute force $O(n^2)$ time is testing every possible pair for duplicates - making n check ops for n items, if the duplicates at the end of the data set it could be terrible performance really $O(n^2)$ checking every pair. We could eleminate the inner loop and turn the algorithm time complexity into $O(n)$ - one pass in total, by leveraging hash map data structure. The trade-off is giving it more space to store n items mapping to their indices. So space turns into $O(n)$ though.
- **Most Resent Index**. As we traverse n items we test the current element if it's in hash map (as key), if yes, get its index and compute the absolute value between the current index. If it falls into the `k` limit - return `True`, otherwise add the current element to tha hash map mapping to its index. Return `False` at the end if no pairs or valid pairs found.

Solid overall shape — hash map storing value → most recent index, $O(n)$ time, $O(n)$ space. But one branch needs precision before we lock in Step 4: when the current value **is** already in the map, but the index distance is **greater** than `k` (check fails) — do you update the map with the current index, or leave the old stored index as-is?

Trace it concretely with `nums = [1, 0, 1, 1]`, `k = 1` using your algorithm exactly as described. What's the correct answer for this input (is there a valid pair)? Now walk through your algorithm step by step — does it get that answer? Pay close attention to what's stored in the map at `i=2` and what gets compared at `i=3`.
- As we progress traversing the input array with $O(n)$ time, we could come across multiple duplicates. If the first pair does not comply the condition (`abs(seen[nums[i]] - i) <= k`), the old stored index has no value anymore - I discard it by overwriting with the current index. This way I build the next index range between the next pair if it exists and check again going along.
Tracing `nums = [1, 0, 1, 1], k = 1`, the valid pair is at indices `2` and `3` (`2-3=-1 -> abs(1) <= 1`), this confirms if the index distance beyond `k`, the hash map's old valud index must be reset to the current one.

Correct — you've identified exactly why the update must happen **unconditionally**:
> **Uncodintional Old Index Update**: the old index is **dead weight once it fails the check**, and keeping it would make you compare future occurrences against a stale, too-far-away position instead of the nearest valid candidate.

Good, that resolves the ambiguity correctly.

**Step 5 — function signature. Step 6 — pseudocode outline reflecting this unconditional-update behavior.**
```py
def contains_nearby_duplicate(nums: list[int], k: int) -> bool:
    # hash map - store {value:index}
    # setup
    # stop early - k == 0: return False
    # set hash map: seen = {}
    # scan the input - for loop - full pass

    # 1. check: if nums[i] in seen & abs(seen[nums[i]] - i) <= k: return True - happy path
    # 2. add current element to seen, mapping to its index
    # 3. expicitly return False at the end - no valid pairs
```
Pseudocode logic is correct — step 2 running unconditionally after step 1 (not nested inside an "else") correctly captures the unconditional index-update behavior you just proved is necessary. The `k==0` early exit is a valid (if optional) optimization given what you derived earlier.

**One thing before Step 7: Wrie code**

Hash map - most-recent index

[Visual](../images/219contains_duplicatesII_hashmap.png)

In [19]:
def contains_nearby_duplicate(nums: list[int], k: int) -> bool:
    # hash map - store value:index
    # setup
    # stop early - k == 0: return False
    # set hash map: seen = {}
    # scan the input - for loop - full pass

    # 1. check: if nums[i] in seen & abs(seen[nums[i]] - i) <= k: return True - happy path
    # 2. add current element to seen, mapping to its index
    # 3. expicitly return False at the end - no valid pairs

    if k == 0: return False

    seen = {}

    for i, value in enumerate(nums):
        # lookup for most recent index & test it
        if value in seen and abs(seen[value] - i) <= k:
            return True
        # add new/update old(stale) index unconditionally
        seen[value] = i

    return False

In [20]:
# contains_nearby_duplicate([1,2,3,1], 3)  # True
contains_nearby_duplicate([1,2,3,4,5], 10)  # False
# contains_nearby_duplicate([1,0,1,1], 1)  # True
# contains_nearby_duplicate([1,2,3,1,2,3], 2)  # False

False

Code looks correct, and the comment earns its place — it documents a non-obvious invariant, not just restating the code. Let's verify with a trace.

**Step 8 — trace this exact code through two cases**: `nums=[1,2,3,1]`, `k=3` (your original example), and `nums=[1,2,3,4,5]`, `k=10` (the no-duplicates counter-example from earlier). Confirm both give the expected result.

**Step 9 — final complexity** confirmation against the actual code: does this match your Step 4 claim of $O(n)$ time / $O(n)$ space? State it in the required format.
- The time complexity is $O(n)$ - one pass + lookup for a key in hash map, matches step 4 claim, and space is $O(k)$ - where k is the number of distinct elements stored in the hash map, since we update indices for stale elements, and hash map doesn't store duplicate keys anyway.

Time complexity confirmed. On space — good instinct to think in terms of `k` (distinct elements) rather than blindly saying `n`, that's the right reflex after Valid Anagram. But apply the same test you just learned there: in Valid Anagram, `k` was capped at 26 by a small fixed alphabet, so `min(n,k)` collapsed to a real constant. Here, values range across `-10⁹` to `10⁹` — is there anything capping the number of _distinct_ values below `n` itself? Can you construct an input where every single element is distinct, forcing `k` to equal `n` exactly?

If so, what's the tightest, honest worst-case space complexity to report here — `O(k)` as a general statement, or does it collapse to something more specific given no smaller alphabet constraint exists in this problem?
- Correct, if all elements are unique we store n items in the hash map, this is exactly `k = n`. The total space complexity will be $O(n)$ for sure.

Confirmed — $O(n)$ time, $O(n)$ space, worst case when all elements are distinct.

**That closes out Contains Duplicate II, and rounds out a solid four-problem run today: Two Sum, Contains Duplicate, Valid Anagram, Contains Duplicate II**.

> **Session debrief — the umbrella pattern:** every one of these traded an $O(n)$ array search for an $O(1)$ hash lookup by **remembering something as you go** — a complement, a membership flag, a frequency count, or a most-recent index. That's the core hash map instinct: whenever brute force needs to ask "*have I seen X?*" or "*does X exist elsewhere in this array?*", a hash structure turns the _asking_ into $O(1)$. Same instinct, four different flavors of "what to store."

The other thread running through today, worth internalizing as a standing habit:
> **stated complexity often hides an assumption.** "$O(1)$ space" assumed a small fixed alphabet in Valid Anagram — broke under Unicode. "Best case" and "worst case" diverged sharply in Contains Duplicate (Counter vs. hash-set, same Big-O, 2500x apart) and in your sliding window recall (per-step vs. amortized removal counts). Before trusting a complexity claim — yours or anyone else's — ask what it's quietly assuming.